# Stage 2 predictor data

This notebook calls the `gold_vol_pipeline.predictors` CLI to download the working `GC=F` gold-futures series and the selected daily predictor panel, align the observations to gold forecast origins, and save a prepared dataset plus run manifest. It writes raw and processed data beneath the Git-ignored `data/` directory.

FRED observations are downloaded as CSV directly from the public FRED website; no API key is required. Yahoo Finance data are also downloaded when the pipeline runs.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / 'pyproject.toml').is_file():
    repo_root = repo_root.parent
if not (repo_root / 'pyproject.toml').is_file():
    raise FileNotFoundError('Could not locate the repository root (pyproject.toml).')

print(f'Repository: {repo_root}')

## Configure the download

The end date is exclusive, so the default avoids requesting today's potentially incomplete bar. Change the range when reproducing a specific analysis snapshot.

In [ ]:
START_DATE = '2015-01-01'
END_DATE = pd.Timestamp.now().normalize().strftime('%Y-%m-%d')
HORIZON = 5
TARGET = 'realized_variance'

print(f'Download range: [{START_DATE}, {END_DATE})')
print(f'Target: {HORIZON}-session forward {TARGET}')

## Download and prepare the panel

The module obtains `GC=F`, DXY, S&P 500, and VIX observations from Yahoo Finance and Treasury yield, breakeven inflation, and WTI observations from FRED. FRED values are aligned using the project's one-business-day availability proxy, not historical release timestamps or vintages.

In [ ]:
command = [
    sys.executable,
    '-m',
    'gold_vol_pipeline.predictors',
    '--start',
    START_DATE,
    '--end',
    END_DATE,
    '--horizon',
    str(HORIZON),
    '--target',
    TARGET,
]
kernel_env = os.environ.copy()
source_path = str(repo_root / 'src')
kernel_env['PYTHONPATH'] = os.pathsep.join(
    path for path in (source_path, kernel_env.get('PYTHONPATH', '')) if path
)
print('Running:', ' '.join(command[1:]))
result = subprocess.run(
    command,
    cwd=repo_root,
    check=True,
    capture_output=True,
    text=True,
    env=kernel_env,
)
print(result.stdout.strip())
if result.stderr.strip():
    print('Downloader messages:\n' + result.stderr.strip())

## Review the generated files

The processed panel retains each predictor's source observation date alongside its availability date. Missing values are not imputed; inspect coverage and freshness before fitting a multivariate model.

In [ ]:
prepared_path = repo_root / 'data/processed/stage2_multivariate.csv'
manifest_path = repo_root / 'data/processed/stage2_multivariate_manifest.json'
prepared = pd.read_csv(prepared_path, parse_dates=['date'])
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

print(f'Prepared rows: {len(prepared):,}')
print(f'Forecast-origin range: {prepared["date"].min().date()} to {prepared["date"].max().date()}')
display(prepared.head())
display(pd.DataFrame(manifest['predictor_sources'].items(), columns=['column', 'source']))